# 00 · Run it all

The whole workflow, end to end: **data → transform → predict → simulate → the latest trades.**
Every step is one call into the project's package. These are the same functions the monthly
scheduled run uses, so this notebook can't drift from what runs unattended. Notebooks 01–06 show
each step in depth.

**No side effects:** running this sends no Telegram message, pushes nothing to Hugging Face, and
never writes the live picks record. It only rebuilds the tables in `data/`.

**Choose the data regime** in the next cell:
- `"local"`: the files already on disk (offline, deterministic; this copy's choice);
- `"hub"`: the private Hugging Face snapshot (needs access);
- `"download"`: fresh data from Wikipedia, Yahoo and FRED.

> Numbers are a **preview** until the download of delisted companies is finished.

In [1]:
REGIME = "local"  # "local" | "hub" | "download"
RETRAIN = False  # True: the full model search and single test (~1 3/4 hours)

import pandas as pd

from sp500_momentum import nb, pipeline

nb.setup()  # caps table output, silences info logs, applies the chart style (see sp500_momentum/nb.py)

## 1. Data

Where the raw data comes from. The delisted companies' monthly prices (Alpha Vantage) are a one-off
download kept on disk or in the snapshot. They never change, so no regime downloads them again.

In [2]:
print(pipeline.get_data(REGIME))

used the files already on disk


## 2. Transform

One row per S&P 500 member per month-end (membership as it was on that date), with past and next-month
returns, then every feature, known at the month-end close, in named sets.

In [3]:
built = pipeline.transform()
sets = built.pop("feature sets")
display(pd.Series(built, name="the monthly table"))
pd.Series(sets, name="columns").to_frame().T

member-months        101431
month-ends              202
first            2010-01-29
latest           2026-10-30
priced               0.9105
Name: the monthly table, dtype: object

,INFO,TO_PREDICT,PAST_RETURNS,TA,STOCK,CROSS_SECTION,MARKET,MACRO,DUMMIES
columns,10,24,5,52,22,17,24,9,23


## 3. Predict

The model's settings were tuned once, on 2014–2022, and are never re-tuned. Here it's **refitted** on
every month whose result is known (the same one-month gap as in the backtest) and scores the latest
month-end. The live system does exactly this each month. `RETRAIN = True` reruns the full search
instead.

In [4]:
if RETRAIN:
    pipeline.retrain()
picks, info = pipeline.latest_picks()
pd.Series(info)

model                               xgb_reg
decision                         2026-10-30
trade at the next open      Mon 02 Nov 2026
refitted on decisions to         2026-08-31
members scored                          503
checks                           all passed
dtype: object

## 4. Simulate

What each strategy did in the backtest, after 10 bps a side: on 2020–2022 (where the headline model
was chosen) and on the untouched test years. These are the same numbers as notebooks 04 and 05 and
the dashboard.

In [5]:
pipeline.backtest_summary().style.format(
    {
        "$10,000 became": "${:,.0f}",
        "yearly return": "{:+.1%}",
        "Sharpe": "{:.2f}",
        "worst fall": "{:.1%}",
    }
)

## 5. The latest trades

Both strategies' picks at the latest month-end, to buy at the next open, equal weight, for one
month. The monthly run publishes the same lists to the Telegram channel and the dashboard.
Research output, not investment advice.

In [6]:
picks.pivot(index="rank", columns="strategy", values="ticker").rename(
    columns={"model": "Headline model", "rule": "Rule: 6-month return"}
)

strategy,Headline model,Rule: 6-month return
rank,,
1,RDDT,MRNA
2,FICO,OKTA
3,APP,CRWD
4,CHTR,CRL
5,SKYD,PANW
6,INTC,DELL
7,SNDK,VEEV
8,INTU,HPE
9,COHR,ILMN


## 6. How it runs unattended

| What | When | Where the results go |
| --- | --- | --- |
| The monthly run (`modeling.predict run`): data → checks → refit → picks | First weekday of each month, 07:00 (Task Scheduler; catch-up if the PC was off) | `reports/picks/`, the Telegram channel, both Hugging Face datasets, the dashboard |
| The Alpha Vantage pull (`dataset av-pull`) | Nightly at 21:00, until the queue is empty | `data/raw/alphavantage/`; a summary to the private chat |
| The run from a snapshot (`modeling.predict run --from-file --no-download`) | Any time, anywhere with access | The same, with no data source touched |

The README lists every command; Chapter 8 explains why each piece is built the way it is.